# 03 - Silver Quality And Clean EDA

Trọng tâm là schema compact, quality checks, quy tắc xử lý null, outlier flags và chỉ báo `is_gold_candidate`.

## Enriched Vs Core Vs Quality Report Vs Clean

- `taxi_weather_trips` là enriched trip-level facts đã join hourly weather và còn giữ thông tin phục vụ truy vết.
- `taxi_weather_trips_core` là fact table rút gọn với kiểu dữ liệu rõ ràng và các validity flags cho outlier.
- `quality_reports/silver_core_quality/latest` ghi nhận kết quả kiểm tra schema, null, outlier và row consistency của Core snapshot.
- `taxi_weather_trips_clean` bổ sung missing flags, clean columns và `is_gold_candidate` sau bước xử lý các field vận hành bị thiếu.

## Input Paths

- `s3a://silver/taxi_weather_trips/`
- `s3a://silver/taxi_weather_trips_core/`
- `s3a://silver/taxi_weather_trips_clean/`
- `s3a://silver/quality_reports/silver_core_quality/latest/`

## Setup Spark Và Utilities


In [1]:
from pathlib import Path
import sys

from pyspark.sql.functions import (
    avg,
    col,
    count as spark_count,
    dayofweek,
    hour,
    lit,
    max as spark_max,
    min as spark_min,
    month,
    stddev,
    sum as spark_sum,
    when,
)

cwd = Path.cwd().resolve()
if (cwd / "utils").exists():
    notebooks_dir = cwd
elif (cwd / "notebooks" / "utils").exists():
    notebooks_dir = cwd / "notebooks"
else:
    notebooks_dir = cwd

if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from utils.spark_session import (
    SILVER_CLEAN_PATH,
    SILVER_CORE_PATH,
    SILVER_QUALITY_PATH,
    SILVER_TAXI_WEATHER_PATH,
    get_spark,
    path_exists,
    safe_display,
    show_schema,
)

spark = get_spark("MetroPulse 03 Silver Quality Clean EDA")
print(f"Spark version: {spark.version}")
print(f"Spark timezone: {spark.conf.get('spark.sql.session.timeZone')}")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/metropulse-notebook-ivy-20260526/cache
The jars for the packages stored in: /tmp/metropulse-notebook-ivy-20260526/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ee388244-2c2a-438a-a3b5-22a562877b78;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central


	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 240ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-ee388244-2c2a-438a-a3b5-22a562877b78
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/6ms)


26/05/26 03:30:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 3.5.1


Spark timezone: America/New_York


## Phạm Vi Output Silver Được Đối Chiếu


In [2]:
dataset_specs = [
    ("enriched", SILVER_TAXI_WEATHER_PATH, "parquet"),
    ("core", SILVER_CORE_PATH, "parquet"),
    ("clean", SILVER_CLEAN_PATH, "parquet"),
    ("quality_report", SILVER_QUALITY_PATH, "json"),
]

datasets = {}

for dataset_name, path, fmt in dataset_specs:
    try:
        if not path_exists(spark, path):
            print(f"Cảnh báo: Không tìm thấy path `{dataset_name}` tại {path}. Bỏ qua dataset này.")
            continue
        datasets[dataset_name] = spark.read.format(fmt).load(path)
        print(f"Loaded {dataset_name}: {path}")
    except Exception as exc:
        print(f"Cảnh báo: Không thể đọc `{dataset_name}` tại {path}. Lý do: {exc}")

enriched_df = datasets.get("enriched")
core_df = datasets.get("core")
clean_df = datasets.get("clean")
quality_df = datasets.get("quality_report")

26/05/26 03:30:30 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Loaded enriched: s3a://silver/taxi_weather_trips/


Loaded core: s3a://silver/taxi_weather_trips_core/


Loaded clean: s3a://silver/taxi_weather_trips_clean/


Loaded quality_report: s3a://silver/quality_reports/silver_core_quality/latest/


## Schema Comparison

So sánh schema cho thấy Core giữ tập cột fact gọn hơn enriched, trong khi Clean bổ sung missing flags, clean columns và `is_gold_candidate` để bảo toàn thông tin missingness sau imputation.

In [3]:
for dataset_name, df in [("enriched", enriched_df), ("core", core_df), ("clean", clean_df)]:
    print(f"\n=== {dataset_name} schema ===")
    if df is None:
        print(f"Cảnh báo: `{dataset_name}` dataset không khả dụng.")
    else:
        show_schema(df)
        safe_display(df.limit(5), n=5, truncate=False)

schema_summary_rows = []
for dataset_name, df in [("enriched", enriched_df), ("core", core_df), ("clean", clean_df)]:
    if df is None:
        schema_summary_rows.append((dataset_name, None, None, "dataset_missing"))
        continue
    for field in df.schema.fields:
        schema_summary_rows.append((dataset_name, field.name, field.dataType.simpleString(), "present"))

if schema_summary_rows:
    schema_summary_df = spark.createDataFrame(
        schema_summary_rows,
        ["dataset_name", "column_name", "data_type", "status"],
    )
    safe_display(schema_summary_df, n=200, truncate=False)


=== enriched schema ===
root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- bronze_ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- vendor_id: long (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecode_id: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pu_location_id: integer (nullable = true)
 |-- do_location_id: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = 

26/05/26 03:30:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------------+---------+-------+-----------------------+--------------------------+-------------------------------+---------+---------------+-------------+-----------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+---------+--------------------+-----------+----------------------+-----------------------+-------------------+-------------------+-------------------+-----------+--------------------------+-------------------+------------------------+----------------+-------------+----------------+----------------+------------+-------------------+--------------+------------------+-------------------+-----------------+---------+
|topic          |partition|offset |kafka_timestamp        |bronze_ingestion_timestamp|source_file                    |vendor_id|passenger_count|trip_distance|ratecode_id|store_and_fwd_flag|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|

+---------+---------+-------------------+-------------------+-------------------+-----------+--------------+--------------+---------------+-------------+-----------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------+----------------+----------------+------------+--------------+------------------+-------------------+-----------------+-------------+---------------------+---------------+--------------------------+-----------------+
|taxi_type|vendor_id|pickup_datetime    |dropoff_datetime   |pickup_hour        |pickup_date|pu_location_id|do_location_id|passenger_count|trip_distance|ratecode_id|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|temperature_f|humidity_percent|precipitation_mm|weather_code|wind_speed_kmh|wind_direction_deg|cloud_cover_percent|is_valid_distance|is_valid_fare|is_valid_total_amount|is_outli

+---------+---------+-------------------+-------------------+-------------------+-----------+--------------+--------------+---------------+-------------+-----------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------+----------------+----------------+------------+--------------+------------------+-------------------+-----------------+-------------+---------------------+---------------+--------------------------+--------------------------+----------------------+-----------------------+-------------------------------+----------------------+---------------------+-----------------+------------------+--------------------------+-----------------+-----------------+--------------------------+-----------------+
|taxi_type|vendor_id|pickup_datetime    |dropoff_datetime   |pickup_hour        |pickup_date|pu_location_id|do_location_id|passenger_count|trip_distance|ratecode_id|payment_type|fare_amount|extra

+------------+-------------------------------+-------------+-------+
|dataset_name|column_name                    |data_type    |status |
+------------+-------------------------------+-------------+-------+
|enriched    |topic                          |string       |present|
|enriched    |partition                      |int          |present|
|enriched    |offset                         |bigint       |present|
|enriched    |kafka_timestamp                |timestamp    |present|
|enriched    |bronze_ingestion_timestamp     |timestamp    |present|
|enriched    |source_file                    |string       |present|
|enriched    |vendor_id                      |bigint       |present|
|enriched    |passenger_count                |double       |present|
|enriched    |trip_distance                  |double       |present|
|enriched    |ratecode_id                    |int          |present|
|enriched    |store_and_fwd_flag             |string       |present|
|enriched    |pu_location_id      

## Evidence: Row Retention Và Gold Candidate Ratio

Phép đối chiếu row count phát hiện một vấn đề lineage quan trọng: enriched hiện có `60,521,651` rows, trong khi Core và Clean đều có `80,922,997` rows, chênh `20,401,346` rows. Vì Core nhiều hơn enriched, ba output này không thuộc cùng snapshot materialization hiện tại. Do đó `78,272,751` candidate rows (`96.72498%` của Clean snapshot) chỉ mô tả Clean đang lưu, không được diễn giải là kết quả đi ra từ enriched output hiện tại.

In [4]:
row_count_results = []
dataset_counts = {}

for dataset_name, df in [("enriched", enriched_df), ("core", core_df), ("clean", clean_df)]:
    if df is None:
        row_count_results.append((dataset_name, None, "dataset_missing"))
    else:
        dataset_counts[dataset_name] = df.count()
        row_count_results.append((dataset_name, dataset_counts[dataset_name], "available"))

row_count_df = spark.createDataFrame(row_count_results, ["dataset_name", "row_count", "status"])
safe_display(row_count_df, n=len(row_count_results), truncate=False)

if "enriched" in dataset_counts and "core" in dataset_counts:
    enriched_rows = dataset_counts["enriched"]
    core_rows = dataset_counts["core"]
    lineage_status = "pass" if enriched_rows == core_rows else "fail_outputs_not_same_snapshot"
    lineage_df = spark.createDataFrame([(
        enriched_rows,
        core_rows,
        core_rows - enriched_rows,
        lineage_status,
    )], ["enriched_rows", "core_rows", "core_minus_enriched_rows", "enriched_to_core_lineage_status"])
    safe_display(lineage_df, n=1, truncate=False)
    if enriched_rows != core_rows:
        print("FAIL: Core/Clean và enriched không thuộc cùng output snapshot. Không diễn giải retention hay Gold candidates là kết quả của Silver enriched hiện tại trước khi rebuild downstream outputs.")

if "core" in dataset_counts and "clean" in dataset_counts:
    core_rows = dataset_counts["core"]
    clean_rows = dataset_counts["clean"]
    retention_df = spark.createDataFrame([(
        core_rows,
        clean_rows,
        core_rows - clean_rows,
        (clean_rows / core_rows) if core_rows else None,
    )], ["core_rows", "clean_rows", "rows_rejected_for_missing_critical_fields", "clean_retention_ratio"])
    safe_display(retention_df, n=1, truncate=False)

if clean_df is None:
    print("Cảnh báo: Không có clean dataset để tính Gold candidate ratio.")
elif "is_gold_candidate" not in clean_df.columns:
    print("Cảnh báo: Clean dataset thiếu `is_gold_candidate`, bỏ qua Gold candidate ratio.")
else:
    candidate_summary = clean_df.agg(
        spark_count(lit(1)).alias("clean_rows"),
        spark_sum(when(col("is_gold_candidate") == True, 1).otherwise(0)).alias("gold_candidate_true_rows"),
        spark_sum(when(col("is_gold_candidate") == False, 1).otherwise(0)).alias("gold_candidate_false_rows"),
        spark_sum(when(col("is_gold_candidate").isNull(), 1).otherwise(0)).alias("gold_candidate_null_rows"),
    )
    candidate_summary = candidate_summary.withColumn(
        "gold_candidate_ratio",
        col("gold_candidate_true_rows") / col("clean_rows"),
    )
    safe_display(candidate_summary, n=1, truncate=False)


[Stage 9:====================>                                    (12 + 4) / 34]




[Stage 9:================================================>        (29 + 4) / 34]




[Stage 12:==============================>                         (13 + 4) / 24]



+------------+---------+---------+
|dataset_name|row_count|status   |
+------------+---------+---------+
|enriched    |60521651 |available|
|core        |80922997 |available|
|clean       |80922997 |available|
+------------+---------+---------+



+-------------+---------+------------------------+-------------------------------+
|enriched_rows|core_rows|core_minus_enriched_rows|enriched_to_core_lineage_status|
+-------------+---------+------------------------+-------------------------------+
|60521651     |80922997 |20401346                |fail_outputs_not_same_snapshot |
+-------------+---------+------------------------+-------------------------------+

FAIL: Core/Clean và enriched không thuộc cùng output snapshot. Không diễn giải retention hay Gold candidates là kết quả của Silver enriched hiện tại trước khi rebuild downstream outputs.


+---------+----------+-----------------------------------------+---------------------+
|core_rows|clean_rows|rows_rejected_for_missing_critical_fields|clean_retention_ratio|
+---------+----------+-----------------------------------------+---------------------+
|80922997 |80922997  |0                                        |1.0                  |
+---------+----------+-----------------------------------------+---------------------+




[Stage 24:===================================>                    (15 + 4) / 24]



+----------+------------------------+-------------------------+------------------------+--------------------+
|clean_rows|gold_candidate_true_rows|gold_candidate_false_rows|gold_candidate_null_rows|gold_candidate_ratio|
+----------+------------------------+-------------------------+------------------------+--------------------+
|80922997  |78272751                |2650246                  |0                       |0.9672497794415598  |
+----------+------------------------+-------------------------+------------------------+--------------------+



## Quality Report Inspection

Quality report hiện ghi nhận `68` checks ở trạng thái `pass` cho Core snapshot đang lưu. Kết quả này chứng minh các rule đã chạy thành công trên Core đó, nhưng không xóa được phát hiện mismatch với enriched snapshot ở phần đối chiếu row count.

In [5]:
if quality_df is None:
    print("Cảnh báo: Quality report path không khả dụng, bỏ qua phần inspection.")
else:
    print("Quality report schema:")
    show_schema(quality_df)
    print("Quality report sample:")
    safe_display(quality_df.limit(20), n=20, truncate=False)

    quality_columns = set(quality_df.columns)
    if "status" in quality_columns:
        print("Quality status summary (proof of passed/failed checks):")
        safe_display(quality_df.groupBy("status").agg(spark_count(lit(1)).alias("check_count")).orderBy("status"), n=10, truncate=False)
        print("Failed checks:")
        failed_df = quality_df.where(col("status") == "fail")
        safe_display(failed_df.limit(100), n=100, truncate=False)
    else:
        print("Quality report thiếu `status`, không thể lọc failed checks.")

    if "check_name" in quality_columns:
        print("Schema checks:")
        safe_display(quality_df.where(col("check_name").startswith("schema_")), n=100, truncate=False)
        print("Null checks:")
        safe_display(
            quality_df.where(
                col("check_name").startswith("critical_null_")
                | col("check_name").startswith("known_nullable_null_")
            ),
            n=100,
            truncate=False,
        )
    else:
        print("Quality report thiếu `check_name`, không thể phân loại schema/null checks.")

Quality report schema:
root
 |-- check_name: string (nullable = true)
 |-- check_value: string (nullable = true)
 |-- checked_at: string (nullable = true)
 |-- dataset_path: string (nullable = true)
 |-- details: string (nullable = true)
 |-- status: string (nullable = true)

Quality report sample:


+----------------------------+-------------+-----------------------------+-------------------------------------+----------------------+------+
|check_name                  |check_value  |checked_at                   |dataset_path                         |details               |status|
+----------------------------+-------------+-----------------------------+-------------------------------------+----------------------+------+
|schema_taxi_type            |string       |2026-05-13T05:32:57.211-04:00|s3a://silver/taxi_weather_trips_core/|expected=string       |pass  |
|schema_vendor_id            |tinyint      |2026-05-13T05:32:57.211-04:00|s3a://silver/taxi_weather_trips_core/|expected=tinyint      |pass  |
|schema_pickup_datetime      |timestamp    |2026-05-13T05:32:57.211-04:00|s3a://silver/taxi_weather_trips_core/|expected=timestamp    |pass  |
|schema_dropoff_datetime     |timestamp    |2026-05-13T05:32:57.211-04:00|s3a://silver/taxi_weather_trips_core/|expected=timestamp    |pass  |

+------+-----------+
|status|check_count|
+------+-----------+
|pass  |68         |
+------+-----------+

Failed checks:


+----------+-----------+----------+------------+-------+------+
|check_name|check_value|checked_at|dataset_path|details|status|
+----------+-----------+----------+------------+-------+------+
+----------+-----------+----------+------------+-------+------+

Schema checks:


+-------------------------------+-------------+-----------------------------+-------------------------------------+----------------------+------+
|check_name                     |check_value  |checked_at                   |dataset_path                         |details               |status|
+-------------------------------+-------------+-----------------------------+-------------------------------------+----------------------+------+
|schema_taxi_type               |string       |2026-05-13T05:32:57.211-04:00|s3a://silver/taxi_weather_trips_core/|expected=string       |pass  |
|schema_vendor_id               |tinyint      |2026-05-13T05:32:57.211-04:00|s3a://silver/taxi_weather_trips_core/|expected=tinyint      |pass  |
|schema_pickup_datetime         |timestamp    |2026-05-13T05:32:57.211-04:00|s3a://silver/taxi_weather_trips_core/|expected=timestamp    |pass  |
|schema_dropoff_datetime        |timestamp    |2026-05-13T05:32:57.211-04:00|s3a://silver/taxi_weather_trips_core/|expected=

+----------------------------------------+-----------+-----------------------------+-------------------------------------+----------------+------+
|check_name                              |check_value|checked_at                   |dataset_path                         |details         |status|
+----------------------------------------+-----------+-----------------------------+-------------------------------------+----------------+------+
|critical_null_taxi_type                 |0          |2026-05-13T05:32:57.211-04:00|s3a://silver/taxi_weather_trips_core/|ratio=0.00000000|pass  |
|critical_null_vendor_id                 |0          |2026-05-13T05:32:57.211-04:00|s3a://silver/taxi_weather_trips_core/|ratio=0.00000000|pass  |
|critical_null_pickup_datetime           |0          |2026-05-13T05:32:57.211-04:00|s3a://silver/taxi_weather_trips_core/|ratio=0.00000000|pass  |
|critical_null_dropoff_datetime          |0          |2026-05-13T05:32:57.211-04:00|s3a://silver/taxi_weather_trips_co

## Null Profile Before And After Cleaning

In [6]:
nullable_columns = [
    "passenger_count",
    "ratecode_id",
    "payment_type",
    "congestion_surcharge",
    "airport_fee",
]
clean_columns = [
    "passenger_count_clean",
    "ratecode_id_clean",
    "payment_type_clean",
    "congestion_surcharge_clean",
    "airport_fee_clean",
]


def null_profile_for_columns(df, dataset_name, columns):
    if df is None:
        return [(dataset_name, column_name, None, None, "dataset_missing") for column_name in columns]
    total_rows = df.count()
    existing_columns = [column_name for column_name in columns if column_name in df.columns]
    missing_rows = [
        (dataset_name, column_name, None, None, "missing_column")
        for column_name in columns
        if column_name not in df.columns
    ]
    if not existing_columns:
        return missing_rows
    null_row = df.agg(
        *[
            spark_sum(when(col(column_name).isNull(), 1).otherwise(0)).alias(column_name)
            for column_name in existing_columns
        ]
    ).collect()[0]
    result_rows = []
    for column_name in existing_columns:
        null_count = int(null_row[column_name] or 0)
        null_ratio = (null_count / total_rows) if total_rows else 0.0
        result_rows.append((dataset_name, column_name, null_count, null_ratio, "present"))
    return result_rows + missing_rows


null_profile_rows = []
null_profile_rows.extend(null_profile_for_columns(core_df, "core", nullable_columns))
null_profile_rows.extend(null_profile_for_columns(clean_df, "clean_original_fields", nullable_columns))
null_profile_rows.extend(null_profile_for_columns(clean_df, "clean_imputed_fields", clean_columns))

null_profile_df = spark.createDataFrame(
    null_profile_rows,
    ["dataset_name", "column_name", "null_count", "null_ratio", "status"],
)
safe_display(null_profile_df, n=len(null_profile_rows), truncate=False)


[Stage 37:>                                                        (0 + 4) / 24]




[Stage 37:====>                                                    (2 + 4) / 24]




[Stage 37:============================>                           (12 + 4) / 24]




[Stage 37:=====================================>                  (16 + 4) / 24]




[Stage 37:==============================================>         (20 + 4) / 24]



...


[Stage 43:=======================================>                (17 + 4) / 24]




[Stage 43:===================================================>    (22 + 2) / 24]




[Stage 49:===================>                                     (8 + 4) / 24]




[Stage 49:=====================================>                  (16 + 4) / 24]



+---------------------+--------------------------+----------+-------------------+-------+
|dataset_name         |column_name               |null_count|null_ratio         |status |
+---------------------+--------------------------+----------+-------------------+-------+
|core                 |passenger_count           |5478285   |0.06769750507386671|present|
|core                 |ratecode_id               |5478285   |0.06769750507386671|present|
|core                 |payment_type              |79941     |9.87865044098651E-4|present|
|core                 |congestion_surcharge      |5478285   |0.06769750507386671|present|
|core                 |airport_fee               |77928015  |0.9629897296067766 |present|
|clean_original_fields|passenger_count           |5478285   |0.06769750507386671|present|
|clean_original_fields|ratecode_id               |5478285   |0.06769750507386671|present|
|clean_original_fields|payment_type              |79941     |9.87865044098651E-4|present|
|clean_ori

## Validate Clean Replacement Logic

In [7]:
replacement_rules = [
    ("passenger_count", "passenger_count_clean", "is_passenger_count_missing", 1),
    ("ratecode_id", "ratecode_id_clean", "is_ratecode_id_missing", 99),
    ("payment_type", "payment_type_clean", "is_payment_type_missing", 0),
    ("congestion_surcharge", "congestion_surcharge_clean", "is_congestion_surcharge_missing", 0.00),
    ("airport_fee", "airport_fee_clean", "is_airport_fee_missing", 0.00),
]

if clean_df is None:
    print("Cảnh báo: Không có clean dataset để validate replacement logic.")
else:
    validation_rows = []
    total_rows = clean_df.count()
    for original_col, clean_col, flag_col, default_value in replacement_rules:
        missing = [column_name for column_name in [original_col, clean_col, flag_col] if column_name not in clean_df.columns]
        if missing:
            validation_rows.append((original_col, clean_col, str(default_value), None, None, None, None, f"missing_columns={missing}"))
            continue

        rule_metrics = clean_df.agg(
            spark_sum(when(col(original_col).isNull(), 1).otherwise(0)).alias("original_null_rows"),
            spark_sum(when(col(flag_col) == True, 1).otherwise(0)).alias("flag_true_rows"),
            spark_sum(when(
                col(original_col).isNull()
                & ((col(flag_col) != True) | col(clean_col).isNull() | (col(clean_col) != lit(default_value))),
                1,
            ).otherwise(0)).alias("imputation_violations"),
            spark_sum(when(
                col(original_col).isNotNull()
                & ((col(flag_col) != False) | (~col(clean_col).eqNullSafe(col(original_col)))),
                1,
            ).otherwise(0)).alias("preservation_violations"),
        ).first()
        imputation_violations = int(rule_metrics["imputation_violations"] or 0)
        preservation_violations = int(rule_metrics["preservation_violations"] or 0)
        status = "pass" if imputation_violations == 0 and preservation_violations == 0 else "fail"
        validation_rows.append(
            (
                original_col,
                clean_col,
                str(default_value),
                int(rule_metrics["original_null_rows"] or 0),
                int(rule_metrics["flag_true_rows"] or 0),
                imputation_violations,
                preservation_violations,
                status,
            )
        )

    validation_df = spark.createDataFrame(
        validation_rows,
        [
            "original_column",
            "clean_column",
            "default_value",
            "original_null_count",
            "missing_flag_true_count",
            "imputation_violations",
            "preservation_violations",
            "status",
        ],
    )
    safe_display(validation_df, n=len(validation_rows), truncate=False)
    print(f"Clean rows checked: {total_rows}")


[Stage 57:===================>                                     (8 + 4) / 24]




[Stage 57:==============================================>         (20 + 4) / 24]




[Stage 60:=========================>                              (11 + 4) / 24]




[Stage 63:===================================>                    (15 + 4) / 24]




[Stage 66:===================>                                     (8 + 4) / 24]




[Stage 66:=====================================>                  (16 + 4) / 24]




[Stage 69:===================>                                     (8 + 4) / 24]



+--------------------+--------------------------+-------------+-------------------+-----------------------+---------------------+-----------------------+------+
|original_column     |clean_column              |default_value|original_null_count|missing_flag_true_count|imputation_violations|preservation_violations|status|
+--------------------+--------------------------+-------------+-------------------+-----------------------+---------------------+-----------------------+------+
|passenger_count     |passenger_count_clean     |1            |5478285            |5478285                |0                    |0                      |pass  |
|ratecode_id         |ratecode_id_clean         |99           |5478285            |5478285                |0                    |0                      |pass  |
|payment_type        |payment_type_clean        |0            |79941              |79941                  |0                    |0                      |pass  |
|congestion_surcharge|congestion_s

## Outlier Profile


In [8]:
profile_df = clean_df if clean_df is not None else core_df
profile_name = "clean" if clean_df is not None else "core"

if profile_df is None:
    print("Cảnh báo: Không có core hoặc clean dataset để tính outlier profile.")
else:
    outlier_exprs = [spark_count(lit(1)).alias("total_rows")]
    if "is_valid_distance" in profile_df.columns:
        outlier_exprs.append(spark_sum(when(~col("is_valid_distance"), 1).otherwise(0)).alias("invalid_distance_rows"))
    if "is_valid_fare" in profile_df.columns:
        outlier_exprs.append(spark_sum(when(~col("is_valid_fare"), 1).otherwise(0)).alias("invalid_fare_rows"))
    if "is_valid_total_amount" in profile_df.columns:
        outlier_exprs.append(spark_sum(when(~col("is_valid_total_amount"), 1).otherwise(0)).alias("invalid_total_amount_rows"))
    if "is_outlier_trip" in profile_df.columns:
        outlier_exprs.append(spark_sum(when(col("is_outlier_trip"), 1).otherwise(0)).alias("outlier_trip_rows"))

    print(f"Outlier profile source: {profile_name}")
    safe_display(profile_df.agg(*outlier_exprs), n=1, truncate=False)

Outlier profile source: clean


+----------+---------------------+-----------------+-------------------------+-----------------+
|total_rows|invalid_distance_rows|invalid_fare_rows|invalid_total_amount_rows|outlier_trip_rows|
+----------+---------------------+-----------------+-------------------------+-----------------+
|80922997  |1627459              |1117083          |990661                   |2650246          |
+----------+---------------------+-----------------+-------------------------+-----------------+




[Stage 74:==============================================>         (20 + 4) / 24]



## Distribution Summaries

Các biến numeric quan trọng được tổng hợp bằng Spark; `approxQuantile` cung cấp phân vị xấp xỉ mà không cần đưa toàn bộ giá trị raw về driver.

In [9]:
distribution_columns = [
    "trip_distance",
    "fare_amount",
    "total_amount",
    "temperature_f",
    "precipitation_mm",
]

if profile_df is None:
    print("Cảnh báo: Không có dataset để tính distribution summaries.")
else:
    available_distribution_columns = [column_name for column_name in distribution_columns if column_name in profile_df.columns]
    missing_distribution_columns = [column_name for column_name in distribution_columns if column_name not in profile_df.columns]
    if missing_distribution_columns:
        print(f"Missing distribution columns: {missing_distribution_columns}")

    summary_rows = []
    for column_name in available_distribution_columns:
        agg_row = profile_df.agg(
            spark_count(col(column_name)).alias("non_null_count"),
            spark_min(col(column_name)).alias("min_value"),
            spark_max(col(column_name)).alias("max_value"),
            avg(col(column_name)).alias("avg_value"),
            stddev(col(column_name)).alias("stddev_value"),
        ).collect()[0]
        quantiles = profile_df.approxQuantile(column_name, [0.25, 0.5, 0.75, 0.95, 0.99], 0.01)
        summary_rows.append(
            (
                column_name,
                agg_row["non_null_count"],
                float(agg_row["min_value"]) if agg_row["min_value"] is not None else None,
                float(agg_row["max_value"]) if agg_row["max_value"] is not None else None,
                float(agg_row["avg_value"]) if agg_row["avg_value"] is not None else None,
                float(agg_row["stddev_value"]) if agg_row["stddev_value"] is not None else None,
                quantiles[0] if len(quantiles) > 0 else None,
                quantiles[1] if len(quantiles) > 1 else None,
                quantiles[2] if len(quantiles) > 2 else None,
                quantiles[3] if len(quantiles) > 3 else None,
                quantiles[4] if len(quantiles) > 4 else None,
            )
        )

    if summary_rows:
        summary_df = spark.createDataFrame(
            summary_rows,
            [
                "column_name",
                "non_null_count",
                "min_value",
                "max_value",
                "avg_value",
                "stddev_value",
                "p25",
                "p50",
                "p75",
                "p95",
                "p99",
            ],
        )
        safe_display(summary_df, n=len(summary_rows), truncate=False)
    else:
        print("Không có distribution columns khả dụng.")


[Stage 77:================================>                       (14 + 4) / 24]




[Stage 80:===========>                                             (5 + 4) / 24]




[Stage 80:=====================>                                   (9 + 4) / 24]




[Stage 80:============================>                           (12 + 4) / 24]




[Stage 80:===================================>                    (15 + 4) / 24]



...


[Stage 100:====================================>                  (16 + 4) / 24]




[Stage 100:=========================================>             (18 + 4) / 24]




[Stage 100:================================================>      (21 + 3) / 24]




[Stage 100:====================================================>  (23 + 1) / 24]



+----------------+--------------+-----------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|column_name     |non_null_count|min_value        |max_value         |avg_value          |stddev_value      |p25               |p50               |p75               |p95               |p99               |
+----------------+--------------+-----------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|trip_distance   |80922997      |0.0              |398608.625        |4.790942651027844  |368.5602658840013 |1.0199999809265137|1.7699999809265137|3.4000000953674316|14.199999809265137|398608.625        |
|fare_amount     |80922997      |-2261.2          |386983.63         |19.37168           |75.59851197337862 |9.3               |13.58             |22.36             |68.8          

## Demand Profile

Demand profile mô tả hình dạng nhu cầu taxi theo giờ trong ngày, ngày trong tuần, tháng và `taxi_type` ở Silver Clean. Đây là phân tích mô tả trên Silver; do Gold chưa triển khai, tôi không trình bày các tổng hợp này như output Gold.

In [10]:
demand_df = clean_df if clean_df is not None else core_df
demand_name = "clean" if clean_df is not None else "core"

if demand_df is None:
    print("Cảnh báo: Không có core hoặc clean dataset để tính demand profile.")
else:
    print(f"Demand profile source: {demand_name}")
    if "pickup_hour" in demand_df.columns:
        print("Trips by hour of day:")
        safe_display(
            demand_df.withColumn("hour_of_day", hour(col("pickup_hour")))
            .groupBy("hour_of_day")
            .agg(spark_count(lit(1)).alias("trip_count"))
            .orderBy("hour_of_day"),
            n=24,
            truncate=False,
        )
    else:
        print("Thiếu `pickup_hour`, bỏ qua trips by hour of day.")

    if "pickup_date" in demand_df.columns:
        print("Trips by day of week:")
        safe_display(
            demand_df.withColumn("day_of_week", dayofweek(col("pickup_date")))
            .groupBy("day_of_week")
            .agg(spark_count(lit(1)).alias("trip_count"))
            .orderBy("day_of_week"),
            n=7,
            truncate=False,
        )
        print("Trips by month:")
        safe_display(
            demand_df.withColumn("pickup_month", month(col("pickup_date")))
            .groupBy("pickup_month")
            .agg(spark_count(lit(1)).alias("trip_count"))
            .orderBy("pickup_month"),
            n=12,
            truncate=False,
        )
    else:
        print("Thiếu `pickup_date`, bỏ qua day-of-week/month demand profile.")

    if "taxi_type" in demand_df.columns:
        print("Trips by taxi_type:")
        safe_display(
            demand_df.groupBy("taxi_type")
            .agg(spark_count(lit(1)).alias("trip_count"))
            .orderBy("taxi_type"),
            n=20,
            truncate=False,
        )
    else:
        print("Thiếu `taxi_type`, bỏ qua trips by taxi_type.")

Demand profile source: clean
Trips by hour of day:



[Stage 104:==>                                                     (1 + 4) / 24]




[Stage 104:===========>                                            (5 + 4) / 24]




[Stage 104:==================>                                     (8 + 4) / 24]




[Stage 104:=========================>                             (11 + 4) / 24]



...


[Stage 113:=========>                                              (4 + 4) / 24]




[Stage 113:======================>                                (10 + 4) / 24]




[Stage 113:=============================>                         (13 + 4) / 24]




[Stage 113:================================================>      (21 + 3) / 24]



+---------+----------+
|taxi_type|trip_count|
+---------+----------+
|green    |1447251   |
|yellow   |79475746  |
+---------+----------+



## Business Interpretation

- Core và Clean hiện cùng có `80,922,997` rows, nhưng không thể nối trực tiếp với enriched `60,521,651` rows vì mismatch snapshot đã được phát hiện rõ trong evidence.
- Quality report có `68` checks pass và các kiểm tra imputation/preservation không ghi nhận vi phạm trên Clean snapshot đang lưu; đây là bằng chứng cho rule cleaning của snapshot đó.
- `airport_fee` có `77,928,015` source null rows, phù hợp với việc green taxi không cung cấp field này; missing flag cần được giữ lại thay vì coi giá trị clean `0.00` là phí nguồn thực tế.
- Outlier flags giữ record đáng nghi cho audit, còn `is_gold_candidate` phân loại `78,272,751` rows sẵn sàng về mặt rule trong Clean snapshot.
- Theo thiết kế handoff hiện tại, Gold sẽ đọc `hourly_weather` và `taxi_weather_trips_core`; tỷ lệ candidate của Clean chỉ là tham chiếu cho policy loại outlier, không phải source contract của Gold và chưa phải bằng chứng của bảng analytics hoặc ML-ready Gold.